In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer

from statsmodels.stats.outliers_influence import variance_inflation_factor

In [2]:
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "Dataset" / "naukri_cleaned.csv"
OUTPUT_PATH = PROJECT_ROOT / "Dataset" / "analytics_jobs.csv"

In [3]:
df = pd.read_csv(DATA_PATH)
df.head()

,uniq_id,crawl_timestamp,job_title,job_salary,job_experience_required,key_skills,role_category,location,functional_area,industry,role,salary_disclosed,salary_min,salary_max,salary_avg,exp_min,exp_max,exp_avg,location_cleaned,key_skills_cleaned
0,9be62c49a0b7ebe982a4af1edaa7bc5f,2019-07-05 01:46:07+00:00,Digital Media Planner,Not Disclosed by Recruiter,5 - 10 yrs,Media Planning| Digital Media,Advertising,Mumbai,"Marketing , Advertising , MR , PR , Media Plan...","Advertising, PR, MR, Event Management",Media Planning Executive/Manager,0,NaN,NaN,NaN,5.0,10.0,7.5,Mumbai,"media planning, digital media"
1,3c52d436e39f596b22519da2612f6a56,2019-07-06 08:04:50+00:00,Online Bidding Executive,Not Disclosed by Recruiter,2 - 5 yrs,pre sales| closing| software knowledge| clien...,Retail Sales,"Pune,Pune","Sales , Retail , Business Development","IT-Software, Software Services",Sales Executive/Officer,0,NaN,NaN,NaN,2.0,5.0,3.5,Pune,"pre sales, closing, software knowledge, client..."
2,ffad8a2396c60be2bf6d0e2ff47c58d4,2019-08-05 15:50:44+00:00,Trainee Research/ Research Executive- Hi- Tec...,Not Disclosed by Recruiter,0 - 1 yrs,Computer science| Fabrication| Quality check|...,R&D,Gurgaon,"Engineering Design , R&D","Recruitment, Staffing",R&D Executive,0,NaN,NaN,NaN,0.0,1.0,0.5,Gurgaon,"computer science, fabrication, quality check, ..."
3,7b921f51b5c2fb862b4a5f7a54c37f75,2019-08-05 15:31:56+00:00,Technical Support,"2,00,000 - 4,00,000 PA.",0 - 5 yrs,Technical Support,Admin/Maintenance/Security/Datawarehousing,Mumbai,"IT Software - Application Programming , Mainte...","IT-Software, Software Services",Technical Support Engineer,1,200000.0,400000.0,300000.0,0.0,5.0,2.5,Mumbai,technical support
4,2d8b7d44e138a54d5dc841163138de50,2019-07-05 02:48:29+00:00,Software Test Engineer -hyderabad,Not Disclosed by Recruiter,2 - 5 yrs,manual testing| test engineering| test cases|...,Programming & Design,Hyderabad,IT Software - QA & Testing,"IT-Software, Software Services",Testing Engineer,0,NaN,NaN,NaN,2.0,5.0,3.5,Hyderabad,"manual testing, test engineering, test cases, ..."


In [4]:
# Keep only Disclosed Salaries

df = df[df["salary_disclosed"] == 1].copy()

In [5]:
lower = df["salary_avg"].quantile(0.01)
upper = df["salary_avg"].quantile(0.99)

df = df[(df["salary_avg"] >= lower) & (df["salary_avg"] <= upper)]

In [6]:
df["log_salary"] = np.log1p(df["salary_avg"])

### Experience Engineering

In [7]:
# Non-linear feature

df["exp_squared"] = df["exp_avg"] ** 2

In [8]:
# Experience Buckets

def exp_bucket(x):
    if x <= 2:
        return "Entry"
    elif x <= 5:
        return "Mid"
    elif x <= 10:
        return "Senior"
    else:
        return "Leadership"

df["exp_bucket"] = df["exp_avg"].apply(exp_bucket)

### City Normalization

In [9]:
# Clean Variants

city_map = {
    "New Delhi": "Delhi",
    "Delhi NCR": "Delhi",
    "Noida": "Delhi",
    "Gurgaon": "Delhi",
    "Navi Mumbai": "Mumbai"
}

df["city_cleaned"] = df["location_cleaned"].replace(city_map)

In [10]:
# Keep Top 7 Cities
top_cities = df["city_cleaned"].value_counts().nlargest(7).index

df["city_group"] = df["city_cleaned"].apply(
    lambda x: x if x in top_cities else "Other"
)

### Industry Cleaning

In [11]:
# Remove Contaminated Long Text
df = df[df["industry"].str.len() < 100]
df["industry"] = df["industry"].str.strip().str.lower()

In [12]:
# Keep Top 10 Industries

top_industries = df["industry"].value_counts().nlargest(10).index

df["industry_group"] = df["industry"].apply(lambda x: x if x in top_industries else "Other")

### Role Grouping

In [13]:
# Create Role Buckets

def role_group(role):
    role = str(role).lower()

    if "developer" in role or "engineer" in role or "architect" in role:
        return "Tech"
    elif "sales" in role or "business development" in role:
        return "Sales"
    elif "hr" in role or "recruit" in role:
        return "HR"
    elif "finance" in role or "account" in role:
        return "Finance"
    elif "manager" in role or "operations" in role:
        return "Operations"
    else:
        return "Other"

df["role_group"] = df["role"].apply(role_group)

### Interaction Features

In [14]:
# Industry x Role 

df["industry_role"] = df["industry_group"] + "_" + df["role_group"]

In [15]:
# Experience x City

df["exp_city_interaction"] = df["exp_avg"] * df["city_group"].astype("category").cat.codes

In [16]:
# Experience × Industry
df["exp_industry_interaction"] = df["exp_avg"] * df["industry_group"].astype("category").cat.codes

### TRAIN / TEST SPLIT BEFORE TF-IDF

In [17]:
# Split
X = df.drop(columns=["log_salary"])
y = df["log_salary"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

In [18]:
X_train["key_skills_cleaned"] = X_train["key_skills_cleaned"].fillna("")
X_test["key_skills_cleaned"] = X_test["key_skills_cleaned"].fillna("")

In [19]:
# Clean skill

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9+.# ]", " ", text)
    return text

X_train["skills_cleaned"] = X_train["key_skills_cleaned"].apply(clean_text)
X_test["skills_cleaned"] = X_test["key_skills_cleaned"].apply(clean_text)

In [20]:
# TF-IDF Fit only on Train

tfidf = TfidfVectorizer(
    max_features=300,
    stop_words="english"
)

train_tfidf = tfidf.fit_transform(X_train["skills_cleaned"])
test_tfidf = tfidf.transform(X_test["skills_cleaned"])

### Categorical Encoding

In [21]:
# One-Hot Encode Grouped Columns

cat_cols = ["exp_bucket", "city_group", "industry_group", "role_group"]

X_train_cat = pd.get_dummies(X_train[cat_cols], drop_first=True)
X_test_cat = pd.get_dummies(X_test[cat_cols], drop_first=True)

X_test_cat = X_test_cat.reindex(columns=X_train_cat.columns, fill_value=0)

In [22]:
# Structured Features 

num_cols = [
    "exp_avg",
    "exp_squared",
    "exp_city_interaction",
    "exp_industry_interaction"
]

X_train_num = X_train[num_cols]
X_test_num = X_test[num_cols]

In [23]:
# Convert TF-IDF to DataFrame

train_tfidf_df = pd.DataFrame(
    train_tfidf.toarray(),
    columns=tfidf.get_feature_names_out()
)

test_tfidf_df = pd.DataFrame(
    test_tfidf.toarray(),
    columns=tfidf.get_feature_names_out()
)

In [24]:
# Final feature Matrix

X_train_final = pd.concat(
    [X_train_num.reset_index(drop=True),
     X_train_cat.reset_index(drop=True),
     train_tfidf_df.reset_index(drop=True)],
    axis=1
)

X_test_final = pd.concat(
    [X_test_num.reset_index(drop=True),
     X_test_cat.reset_index(drop=True),
     test_tfidf_df.reset_index(drop=True)],
    axis=1
)

In [25]:
df.columns

Index(['uniq_id', 'crawl_timestamp', 'job_title', 'job_salary',
       'job_experience_required', 'key_skills', 'role_category', 'location',
       'functional_area', 'industry', 'role', 'salary_disclosed', 'salary_min',
       'salary_max', 'salary_avg', 'exp_min', 'exp_max', 'exp_avg',
       'location_cleaned', 'key_skills_cleaned', 'log_salary', 'exp_squared',
       'exp_bucket', 'city_cleaned', 'city_group', 'industry_group',
       'role_group', 'industry_role', 'exp_city_interaction',
       'exp_industry_interaction'],
      dtype='object')

In [26]:
analytics_df = df[[
    "uniq_id",
    "job_title",
    "industry",
    "industry_group",
    "role",
    "role_group",
    "industry_role",
    "city_cleaned",
    "city_group",
    "salary_disclosed",
    "salary_min",
    "salary_max",
    "salary_avg",
    "exp_avg",
    "exp_bucket",
    "key_skills_cleaned"
]].copy()

analytics_df.to_csv(OUTPUT_PATH, index=False)

In [27]:
import joblib

# Save datasets
joblib.dump(X_train_final, PROJECT_ROOT / "Dataset" / "X_train.pkl")
joblib.dump(X_test_final, PROJECT_ROOT / "Dataset" / "X_test.pkl")
joblib.dump(y_train, PROJECT_ROOT / "Dataset" / "y_train.pkl")
joblib.dump(y_test, PROJECT_ROOT / "Dataset" / "y_test.pkl")

# Save TF-IDF vectorizer
joblib.dump(tfidf, PROJECT_ROOT / "Dataset" / "tfidf_vectorizer.pkl")

print("Feature engineering artifacts saved successfully.")

Feature engineering artifacts saved successfully.
